# Notebook 1: Environment Validation + Occlusion Test

**Purpose:** Verify that vLLM runs on Jetson Thor with the SM100 workaround, loads a VLM,
and can describe a Monopoly board image. Then run the occlusion gate on your 20 test photos.

**Gate:** If occupied-square false-negative rate >= 5%, STOP. Investigate camera mount before
running Notebook 2. Time-box camera fix to 2 hours. If not resolved, the single-image premise
needs revision.

**SM100 bug note (vLLM issue #38411):** All multimodal vision models crash at startup on
Jetson Thor (SM100/Blackwell) without the workaround flags. `--skip-mm-profiling` and
`--attention-backend TRITON_ATTN` are required. Image inference works normally after startup.

In [ ]:
# ---- CELL 1: Environment check ----
# Run this first. Pin versions after your first successful run.
import subprocess, sys

required = [
    'openai',
    'Pillow',
    'scikit-learn',
    'numpy',
    'tqdm',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', '../requirements.txt'])

import importlib, pkg_resources
print('=== Environment ===')
print(f'Python: {sys.version}')
for pkg in required:
    try:
        ver = pkg_resources.get_distribution(pkg).version
        print(f'  {pkg}: {ver}')
    except Exception:
        print(f'  {pkg}: NOT FOUND')

# Print vLLM version if installed
try:
    import vllm
    print(f'  vllm: {vllm.__version__}')
    # TODO: paste the known-good vLLM commit here after Day 1 succeeds
    # KNOWN_GOOD_VLLM_COMMIT = "abc1234"
except ImportError:
    print('  vllm: not importable as Python module (expected if running as server only)')

print('\nDone.')

In [ ]:
# ---- CELL 2: Configuration ----
# Edit these before running the rest of the notebook.

import os
from pathlib import Path

# Model to validate in this notebook (Qwen2.5-VL is recommended first candidate)
# Verify this ID on https://huggingface.co/Qwen before Day 0 download
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct-AWQ"

# vLLM server settings
VLLM_HOST = "localhost"
VLLM_PORT = 8000
VLLM_BASE_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1"

# Data directories
IMAGES_DIR = Path("../data/images")        # JPEGs from camera
GT_DIR     = Path("../data/ground_truth")  # board_001_gt.json files

# Occlusion gate threshold
FNR_THRESHOLD = 0.05  # 5% — if exceeded, STOP and investigate camera

print(f'Model:      {MODEL_ID}')
print(f'vLLM URL:   {VLLM_BASE_URL}')
print(f'Images dir: {IMAGES_DIR} (exists: {IMAGES_DIR.exists()})')
print(f'GT dir:     {GT_DIR} (exists: {GT_DIR.exists()})')
print(f'FNR gate:   {FNR_THRESHOLD:.0%}')

In [ ]:
# ---- CELL 3: Launch vLLM with SM100 workaround ----
#
# SM100 workaround flags (vLLM issue #38411 — required on Jetson Thor Blackwell):
#   --skip-mm-profiling        bypasses vision encoder memory profiling pass
#   --attention-backend TRITON_ATTN  uses Triton instead of SM80-compiled FA2 kernel
#
# Image inference works normally after startup with these flags.

import subprocess, time, requests, signal

VLLM_CMD = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_ID,
    '--skip-mm-profiling',              # SM100 workaround
    '--attention-backend', 'TRITON_ATTN',  # SM100 workaround
    '--dtype', 'auto',
    '--host', VLLM_HOST,
    '--port', str(VLLM_PORT),
    # Uncomment for trust-remote-code models (VILA):
    # '--trust-remote-code',
]

def wait_for_vllm(url, timeout_s=300, poll_s=5):
    """Poll /health until vLLM is ready or timeout."""
    health_url = url.replace('/v1', '/health')
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            r = requests.get(health_url, timeout=2)
            if r.status_code == 200:
                return True
        except Exception:
            pass
        print(f'  waiting for vLLM... ({int(deadline - time.time())}s left)', end='\r')
        time.sleep(poll_s)
    return False

print(f'Launching vLLM: {" ".join(VLLM_CMD)}')
print('(startup takes 30-120s for model load)')

vllm_proc = subprocess.Popen(
    VLLM_CMD,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

ready = wait_for_vllm(VLLM_BASE_URL, timeout_s=300)
if ready:
    print('\nvLLM is ready.')
else:
    vllm_proc.terminate()
    raise RuntimeError(
        'vLLM did not become ready within 5 minutes.\n'
        'Check vllm_proc.stdout for crash details.\n'
        'Common cause: SM100 crash — verify workaround flags are present.'
    )

In [ ]:
# ---- CELL 4: Image helper ----
# Loads a JPEG and encodes it as base64 for the OpenAI-compatible vision API.
# This format works uniformly across Qwen2.5-VL, VILA, LLaVA, and Cosmos Nemotron.

import base64
from PIL import Image
import io

def image_to_b64(image_path: Path, max_size: int = 1024) -> str:
    """Load image, resize to max_size on longest edge, return base64 JPEG string."""
    img = Image.open(image_path).convert('RGB')
    # Resize to keep inference time reasonable without losing board detail
    w, h = img.size
    if max(w, h) > max_size:
        scale = max_size / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=90)
    return base64.b64encode(buf.getvalue()).decode('utf-8')

def make_vision_message(b64_image: str, prompt: str) -> list:
    """Build OpenAI-compatible chat message with image and text."""
    return [
        {
            'role': 'user',
            'content': [
                {
                    'type': 'image_url',
                    'image_url': {'url': f'data:image/jpeg;base64,{b64_image}'},
                },
                {
                    'type': 'text',
                    'text': prompt,
                },
            ],
        }
    ]

print('Image helpers ready.')

In [ ]:
# ---- CELL 5: Single inference smoke test ----
# Pick any image from data/images/ and verify the model responds.
# If this cell passes, the SM100 workaround is working and the model loaded.

from openai import OpenAI

client = OpenAI(base_url=VLLM_BASE_URL, api_key='not-required')

# Find first image in IMAGES_DIR
test_images = sorted(IMAGES_DIR.glob('*.jpg')) + sorted(IMAGES_DIR.glob('*.jpeg'))
if not test_images:
    raise FileNotFoundError(
        f'No images found in {IMAGES_DIR}.\n'
        'Complete Day 0: photograph the board and place JPEGs in data/images/.'
    )

test_img_path = test_images[0]
print(f'Using test image: {test_img_path.name}')

b64 = image_to_b64(test_img_path)
messages = make_vision_message(
    b64,
    prompt='Describe what you see on this Monopoly board. List which squares have pieces or buildings on them.',
)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=256,
    temperature=0.0,
)

answer = response.choices[0].message.content
print(f'\nModel response:\n{answer}')
print('\nSmoke test PASSED — model loaded and returned a response.')

In [ ]:
# ---- CELL 6: Load ground truth for occlusion test ----
# GT file format: board_001_gt.json
# Schema: {"pieces": [{"square_id": "06", "piece_type": "token_blue", "owner": "player_1"}]}
# Only OCCUPIED squares are listed. Absence = unoccupied.

import json

VALID_SQUARE_IDS = {f'{i:02d}' for i in range(40)}
VALID_PIECE_TYPES = {'house', 'hotel'} | {f'token_{c}' for c in ['red','blue','green','yellow']}
# Add other token colors your set uses:
# VALID_PIECE_TYPES |= {'token_purple', 'token_white'}

def load_ground_truth(gt_dir: Path) -> dict:
    """Load all *_gt.json files. Returns {image_stem: list_of_pieces}."""
    gt_data = {}
    for gt_file in sorted(gt_dir.glob('*_gt.json')):
        with open(gt_file) as f:
            data = json.load(f)
        pieces = data.get('pieces', [])
        # Integrity check
        square_ids_seen = set()
        for p in pieces:
            sid = p.get('square_id', '')
            if sid not in VALID_SQUARE_IDS:
                raise ValueError(f'{gt_file.name}: invalid square_id "{sid}" (must be "00"–"39")')
            if sid in square_ids_seen:
                raise ValueError(f'{gt_file.name}: duplicate square_id "{sid}"')
            square_ids_seen.add(sid)
            pt = p.get('piece_type', '')
            if pt not in VALID_PIECE_TYPES:
                raise ValueError(
                    f'{gt_file.name}: unknown piece_type "{pt}".\n'
                    f'Valid types: {sorted(VALID_PIECE_TYPES)}'
                )
        image_stem = gt_file.stem.replace('_gt', '')
        gt_data[image_stem] = pieces
    return gt_data

gt_data = load_ground_truth(GT_DIR)
print(f'Loaded {len(gt_data)} ground truth files.')
if len(gt_data) == 0:
    raise FileNotFoundError(
        f'No *_gt.json files found in {GT_DIR}.\n'
        'Complete Day 0: label board images and place GT files in data/ground_truth/.'
    )
total_pieces = sum(len(v) for v in gt_data.values())
print(f'Total occupied squares across all images: {total_pieces}')
print('\nGround truth integrity check PASSED.')

In [ ]:
# ---- CELL 7: Occlusion gate ----
# Ask the VLM to list all occupied squares in each image.
# Measure false-negative rate (occupied square not detected by VLM).
# PASS: FNR < 5%  → proceed to Notebook 2
# FAIL: FNR >= 5% → investigate camera mount (time-box to 2 hours, then escalate)

from tqdm import tqdm

OCCLUSION_PROMPT = (
    'Look at this Monopoly board image. '
    'List every square that has a piece or building on it. '
    'For each occupied square, give the square number (00=Go, 39=Boardwalk) '
    'and describe what is on it. '
    'If a square is empty, do not mention it.'
)

results = []  # {image, gt_squares, detected_text, fn_squares}

for image_stem, gt_pieces in tqdm(gt_data.items(), desc='Occlusion test'):
    # Find matching image file
    img_candidates = list(IMAGES_DIR.glob(f'{image_stem}.*'))
    if not img_candidates:
        print(f'  WARNING: no image found for GT file {image_stem} — skipping')
        continue

    img_path = img_candidates[0]
    b64 = image_to_b64(img_path)
    messages = make_vision_message(b64, OCCLUSION_PROMPT)

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        max_tokens=512,
        temperature=0.0,
    )
    detected_text = response.choices[0].message.content

    # Check which GT squares appear (by square_id) in the response text
    gt_square_ids = {p['square_id'] for p in gt_pieces}
    fn_squares = set()
    for sid in gt_square_ids:
        # Accept if the square_id number (with or without leading zero) appears in text
        if sid not in detected_text and str(int(sid)) not in detected_text:
            fn_squares.add(sid)

    results.append({
        'image': image_stem,
        'gt_count': len(gt_square_ids),
        'fn_count': len(fn_squares),
        'fn_squares': sorted(fn_squares),
        'detected_text': detected_text,
    })

# Compute overall FNR
total_gt = sum(r['gt_count'] for r in results)
total_fn = sum(r['fn_count'] for r in results)
fnr = total_fn / total_gt if total_gt > 0 else 0.0

print(f'\n=== Occlusion Gate Results ===')
print(f'Images tested:    {len(results)}')
print(f'Total GT squares: {total_gt}')
print(f'Missed (FN):      {total_fn}')
print(f'FNR:              {fnr:.1%}')

if fnr < FNR_THRESHOLD:
    print(f'\nOCCLUSION GATE: PASS ({fnr:.1%} < {FNR_THRESHOLD:.0%})')
    print('Proceed to Notebook 2.')
else:
    print(f'\nOCCLUSION GATE: FAIL ({fnr:.1%} >= {FNR_THRESHOLD:.0%})')
    print('Action: investigate camera mount position.')
    print('Time-box to 2 hours. If not resolved, the single-image premise needs revision.')
    print('\nWorst offenders (most FNs):')
    for r in sorted(results, key=lambda x: x['fn_count'], reverse=True)[:5]:
        print(f'  {r["image"]}: {r["fn_count"]} missed squares {r["fn_squares"]}')

In [ ]:
# ---- CELL 8: Shutdown vLLM ----
# Run this when done with Notebook 1 to free GPU memory before Notebook 2.

import time

if 'vllm_proc' in dir() and vllm_proc.poll() is None:
    print('Terminating vLLM process...')
    vllm_proc.terminate()
    vllm_proc.wait(timeout=30)
    print('vLLM stopped.')

    # Wait for GPU memory to clear (Jetson unified memory)
    print('Waiting for GPU memory to free...')
    deadline = time.time() + 120
    while time.time() < deadline:
        try:
            out = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                text=True
            ).strip()
            used_mb = int(out.split('\n')[0])
            print(f'  GPU memory used: {used_mb} MB', end='\r')
            if used_mb < 2048:  # < 2GB = clear
                break
        except Exception:
            break  # nvidia-smi not available or unified memory — proceed
        time.sleep(5)
    print('\nGPU memory free. Ready for Notebook 2.')
else:
    print('vLLM process not running.')